## 1. Setup e carga dos dados

**Objetivo:** importar as bibliotecas necessárias e carregar a base bruta (`desafio_nps_fase_1.csv`) para iniciar o tratamento.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

# Caminho para a base de dados
    
data = '../data/raw/desafio_nps_fase_1.csv'
df = pd.read_csv(data)

print(f'Dimensão da base: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

Dimensão da base: 2500 linhas x 19 colunas


,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


### Estrutura da base

**Objetivo:** conferir tipos de dados de cada coluna (numérico, texto, etc.) e o total de linhas/colunas, para identificar colunas que possam precisar de conversão de tipo.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   customer_id                2500 non-null   int64  
 1   customer_age               2500 non-null   int64  
 2   customer_region            2500 non-null   object 
 3   customer_tenure_months     2500 non-null   int64  
 4   order_id                   2500 non-null   int64  
 5   order_value                2500 non-null   float64
 6   items_quantity             2500 non-null   int64  
 7   discount_value             2500 non-null   float64
 8   payment_installments       2500 non-null   int64  
 9   delivery_time_days         2500 non-null   int64  
 10  delivery_delay_days        2500 non-null   int64  
 11  freight_value              2500 non-null   float64
 12  delivery_attempts          2500 non-null   int64  
 13  customer_service_contacts  2500 non-null   int64

### Verificação de valores nulos

**Objetivo:** identificar se existem campos vazios (NaN) em alguma coluna, o que exigiria decisão de preenchimento ou remoção.

**Resultado:** nenhum valor nulo encontrado na base.

In [4]:
nulos = df.isnull().sum()
print('Valores nulos por coluna:')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Nenhum valor nulo encontrado.')

Valores nulos por coluna:
Nenhum valor nulo encontrado.


### Verificação de duplicados

**Objetivo:** checar se existem linhas inteiras repetidas, ou `customer_id`/`order_id` duplicados (o que indicaria erro de carga de dados).

**Resultado:** nenhuma duplicata encontrada.

In [5]:
print(f'Linhas totalmente duplicadas: {df.duplicated().sum()}')
print(f'customer_id duplicados: {df["customer_id"].duplicated().sum()}')
print(f'order_id duplicados: {df["order_id"].duplicated().sum()}')

Linhas totalmente duplicadas: 0
customer_id duplicados: 0
order_id duplicados: 0


### Estatísticas descritivas

**Objetivo:** olhar min, max, média e desvio-padrão de cada coluna numérica, para identificar valores fora do plausível (ex: idade negativa, NPS fora de 0-10).

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,2500.0,1250.500000,721.832160,1.00,625.7500,1250.500,1875.2500,2500.00
customer_age,2500.0,43.396000,14.888487,18.00,31.0000,43.000,56.0000,69.00
customer_tenure_months,2500.0,61.322400,34.478729,1.00,31.0000,62.000,91.0000,119.00
order_id,2500.0,51250.500000,721.832160,50001.00,50625.7500,51250.500,51875.2500,52500.00
order_value,2500.0,434.259740,289.772497,7.76,220.2450,375.515,577.2900,1983.81
items_quantity,2500.0,3.470800,1.687331,1.00,2.0000,3.000,5.0000,6.00
discount_value,2500.0,29.745620,29.225603,0.02,8.8850,20.935,40.8325,230.33
payment_installments,2500.0,6.004000,3.159743,1.00,3.0000,6.000,9.0000,11.00
delivery_time_days,2500.0,8.022000,3.770411,2.00,5.0000,8.000,11.0000,14.00
delivery_delay_days,2500.0,2.187200,1.454442,0.00,1.0000,2.000,3.0000,8.00


### Validação de regras de negócio

**Objetivo:** rodar checagens específicas do domínio do problema (ex: `nps_score` só pode ir de 0 a 10, `discount_value` não pode ser maior que `order_value`), que estatísticas genéricas (describe) não pegam sozinhas.

**Resultado:** encontramos 35 registros (1,4% da base) com `discount_value` maior que `order_value`.

In [7]:
inconsistencias = {
    'nps_score fora de 0-10': ((df['nps_score'] < 0) | (df['nps_score'] > 10)).sum(),
    'customer_age fora de 0-100': ((df['customer_age'] < 0) | (df['customer_age'] > 100)).sum(),
    'order_value negativo ou zero': (df['order_value'] <= 0).sum(),
    'discount_value maior que order_value': (df['discount_value'] > df['order_value']).sum(),
    'items_quantity <= 0': (df['items_quantity'] <= 0).sum(),
    'delivery_time_days negativo': (df['delivery_time_days'] < 0).sum(),
    'delivery_delay_days negativo': (df['delivery_delay_days'] < 0).sum(),
    'customer_tenure_months negativo': (df['customer_tenure_months'] < 0).sum(),
    'repeat_purchase_30d fora de {0,1}': (~df['repeat_purchase_30d'].isin([0, 1])).sum(),
}

for regra, qtd in inconsistencias.items():
    status = '⚠️' if qtd > 0 else '✅'
    print(f'{status} {regra}: {qtd} registros')

✅ nps_score fora de 0-10: 0 registros
✅ customer_age fora de 0-100: 0 registros
✅ order_value negativo ou zero: 0 registros
⚠️ discount_value maior que order_value: 35 registros
✅ items_quantity <= 0: 0 registros
✅ delivery_time_days negativo: 0 registros
✅ delivery_delay_days negativo: 0 registros
✅ customer_tenure_months negativo: 0 registros
✅ repeat_purchase_30d fora de {0,1}: 0 registros


### Investigação da inconsistência de desconto

**Objetivo:** entender a magnitude do problema antes de decidir como tratar — ver se é erro de arredondamento (diferença pequena) ou erro grave de sistema (diferença grande).

**Achado:** a diferença varia de ~1,1x até ~3,9x o valor do pedido — não é arredondamento, é erro de dado real.

In [8]:
inconsistentes = df[df['discount_value'] > df['order_value']]
print(f'Total de casos: {len(inconsistentes)}')
inconsistentes[['customer_id', 'order_id', 'order_value', 'discount_value']].sort_values('discount_value', ascending=False)

Total de casos: 35


,customer_id,order_id,order_value,discount_value
1404,1405,51405,68.20,197.60
1777,1778,51778,110.52,181.97
1363,1364,51364,68.85,140.73
270,271,50271,79.93,129.11
1693,1694,51694,107.37,121.72
1096,1097,51097,112.19,119.93
947,948,50948,97.74,115.67
57,58,50058,86.15,112.42
2456,2457,52457,30.68,99.74
6,7,50007,41.29,99.62


### Tratamento: discount_value > order_value

Foram identificados 35 registros (1,4% da base) onde o valor do desconto era maior que o valor do pedido — 
inconsistência não plausível no mundo real. A discrepância variava de ~1,1x até ~3,9x o valor do pedido, 
descartando erro de arredondamento e sugerindo falha de sistema/digitação na origem dos dados.

**Decisão:** essas 35 linhas foram removidas da base, por representarem uma fração pequena dos dados e por 
não haver como inferir com confiança o valor correto do desconto.
documentando a decisão — importante pro README/notebook

In [9]:
qtd_antes = len(df)
df = df[df['discount_value'] <= df['order_value']].reset_index(drop=True)
qtd_depois = len(df)

print(f'Registros removidos: {qtd_antes - qtd_depois}')
print(f'Base antes: {qtd_antes} linhas | Base depois: {qtd_depois} linhas')

Registros removidos: 35
Base antes: 2500 linhas | Base depois: 2465 linhas


### Revalidação pós-tratamento

**Objetivo:** confirmar que a inconsistência foi realmente eliminada após a remoção.

In [10]:
qtd = (df['discount_value'] > df['order_value']).sum()
print(f'discount_value > order_value: {qtd} registros' + (' ✅' if qtd == 0 else ' ⚠️'))

discount_value > order_value: 0 registros ✅


### Verificação de categorias de texto

**Objetivo:** conferir se os valores da coluna `customer_region` estão padronizados (sem variações de escrita tipo "Sul"/"sul"/"SUL" que fragmentariam a análise por região).

In [11]:
print(df['customer_region'].value_counts())

customer_region
Sudeste         514
Sul             513
Norte           502
Nordeste        475
Centro-Oeste    461
Name: count, dtype: int64


### Estatísticas descritivas

**Objetivo:** olhar min, max, média, desvio-padrão e quartis de cada coluna numérica, para identificar valores fora do plausível (ex: idade negativa, NPS fora de 0-10, valores de pedido ou frete negativos) que não seriam pegos apenas checando nulos e duplicados.

**Resultado:** todas as colunas estão dentro de faixas plausíveis — idades entre 18 e 69 anos, valores de pedido positivos, quantidade de itens entre 1 e 6, sem atrasos negativos. Essa análise complementa a validação de regras de negócio feita na célula seguinte, que checa violações específicas (como `discount_value` maior que `order_value`).

In [13]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
customer_id,2465.0,1253.188641,721.280948,1.00,629.00,1254.00,1879.00,2500.00
customer_age,2465.0,43.324949,14.899865,18.00,31.00,43.00,56.00,69.00
customer_tenure_months,2465.0,61.298174,34.490505,1.00,31.00,62.00,91.00,119.00
order_id,2465.0,51253.188641,721.280948,50001.00,50629.00,51254.00,51879.00,52500.00
order_value,2465.0,439.715611,288.136556,14.58,225.35,379.89,584.77,1983.81
items_quantity,2465.0,3.472617,1.686992,1.00,2.00,3.00,5.00,6.00
discount_value,2465.0,29.042767,28.406362,0.02,8.72,20.51,40.03,230.33
payment_installments,2465.0,6.004462,3.159514,1.00,3.00,6.00,9.00,11.00
delivery_time_days,2465.0,8.024746,3.771125,2.00,5.00,8.00,11.00,14.00
delivery_delay_days,2465.0,2.188641,1.455130,0.00,1.00,2.00,3.00,8.00


### Exportação da base tratada

**Objetivo:** salvar a versão limpa dos dados em `data/processed/`, separada da base bruta em `data/raw/`, para que os próximos notebooks (EDA e modelagem) usem sempre a versão tratada.

In [14]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/desafio_nps_fase_1_clean.csv', index=False)
print(f'Base tratada salva em data/processed/ com {len(df)} linhas.')

Base tratada salva em data/processed/ com 2465 linhas.


## Resumo do tratamento de dados

- Base original: 2.500 registros, 19 colunas
- Valores nulos: nenhum encontrado
- Linhas duplicadas: nenhuma encontrada
- Inconsistência identificada: 35 registros (1,4%) com discount_value > order_value → removidos
- Base final: registros restantes após limpeza, salva em `data/processed/desafio_nps_fase_1_clean.csv`

## Resumo do tratamento de dados

| Verificação | Resultado |
|---|---|
| Base original | 2.500 linhas, 19 colunas |
| Valores nulos | Nenhum |
| Linhas duplicadas | Nenhuma |
| `discount_value > order_value` | 35 registros (1,4%) → removidos |
| Base final | 2.465 linhas |
| Arquivo de saída | `data/processed/desafio_nps_fase_1_clean.csv` |
